In [124]:
# Load libraries, set constant values
import os
from matplotlib import pyplot as plt
import pandas as pd

import torch
import torchaudio
import torch.nn as nn
import numpy as np
import librosa
import torchvision.datasets
import torch.nn.functional as F
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from birdclef_utils.datasets import collate_windowed, build_windowed_loader, compute_window_weights_from_class_weights, log_epoch_coverage
from torch.nn.modules.flatten import Flatten
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

DATA_DIR='data/'
TEST_SOUNDSCAPES_DIR='test_soundscapes/'
TRAIN_SOUNDSCAPES_DIR='train_soundscapes/'
TRAIN_AUDIO_DIR='train_audio/'

SOUNDSCAPES_DESCRIPTION_FILE=os.path.join(DATA_DIR, 'train_soundscapes_labels.csv')
AUDIO_DESCRIPTION_FILE=os.path.join(DATA_DIR, 'train.csv')
TAXONOMY_FILE=os.path.join(DATA_DIR, 'taxonomy.csv')

SAMPLE_TRAIN_AUDIO_FILE_PATH=os.path.join(DATA_DIR, TRAIN_AUDIO_DIR, '22930', 'iNat317238.ogg')
SAMPLE_TRAIN_SOUNDSCAPES_FILE_PATH=os.path.join(DATA_DIR, TRAIN_SOUNDSCAPES_DIR, 'BC2026_Train_0001_S08_20250606_030007.ogg')

training_file_path = os.path.join(DATA_DIR, TRAIN_AUDIO_DIR)
test_soundscapes_path = os.path.join(DATA_DIR, TEST_SOUNDSCAPES_DIR)
train_soundscapes_path = os.path.join(DATA_DIR, TRAIN_SOUNDSCAPES_DIR)

# Global verbosity control for notebook print output and training logs.
VERBOSE = True
# Controls training-specific logs (model.summary, epoch progress, callbacks).
# Kept separate from VERBOSE so training output is visible even when status prints are off.
TRAIN_VERBOSE = True

# Quick-run defaults (can be overridden later)
QUICK_RUN = False
QUICK_RUN_SUBSET = 1000       # training rows per run
QUICK_RUN_VAL_SUBSET = 300    # validation rows per run
QUICK_RUN_EPOCHS = 3         # epochs per quick-run

# Resume controls (v2): extend a completed run from a saved checkpoint.
# Keeps default behavior unchanged unless RESUME_TRAINING=True.
RESUME_TRAINING = True
RESUME_CHECKPOINT_PATH = 'checkpoints/best_model_20260603_231706.pt'
RESUME_INITIAL_EPOCH = 20
RESUME_EXTRA_EPOCHS = 3

# Threshold sweep tuning knobs (Cell 13).
# Use these to control speed vs quality without editing Cell 13 directly.
FAST_SWEEP_MAX_ROWS = 200
FAST_SWEEP_MAX_CROPS = 2
FAST_SWEEP_THRESHOLD_START = 0.10
FAST_SWEEP_THRESHOLD_STOP = 0.95
FAST_SWEEP_THRESHOLD_STEP = 0.10
FULL_SWEEP_MAX_ROWS = 1000
FULL_SWEEP_THRESHOLD_START = 0.05
FULL_SWEEP_THRESHOLD_STOP = 0.95
FULL_SWEEP_THRESHOLD_STEP = 0.05

# Soundscape augmentation: add 5-second labeled windows from train_soundscapes_labels.csv.
USE_TRAIN_SOUNDSCAPE_LABELS = True

# Use full files; set clipping via RANDOM_CROP_SECONDS instead of hard truncating to first N seconds.
MAX_AUDIO_SECONDS = None
RANDOM_CROP_SECONDS = 10  # Reduced from 40 for faster training (~1000 timesteps vs ~4000)

# Feature extraction: enable log-mel features.
USE_LOG_MEL = True
STFT_FRAME_LENGTH = 1024
STFT_FRAME_STEP = 320
STFT_FFT_LENGTH = 1024
N_MELS = 128
MEL_FMIN = 50.0
MEL_FMAX = 14000.0

# Spectrogram caching: 3-5x speedup (first epoch slower while caching, subsequent epochs much faster)
USE_SPECTROGRAM_CACHE = True
SPECTROGRAM_CACHE_DIR = 'cache/spectrograms'

USE_FOCAL_LOSS = True
FOCAL_GAMMA = 2.0

# Validation design: average predictions over multiple random crops for stable eval.
VALIDATION_NUM_CROPS = 3

# Number of rows to use during validation threshold evaluations (default 1000)
VALIDATION_EVAL_ROWS = globals().get('VALIDATION_EVAL_ROWS', 1000)

# Default full-run epochs if not overridden
EPOCHS = 10

Device: mps


In [125]:
## Cell 2
import importlib
import logging
import birdclef_utils as birdclef_utils_pkg
import birdclef_utils.audio_processing as birdclef_audio_processing
import birdclef_utils.metrics as birdclef_metrics
import birdclef_utils.datasets as birdclef_datasets
import birdclef_utils.models as birdclef_models
from birdclef_utils import load_audio_ogg

# Reload utility modules so notebook picks up local code edits without kernel restart.
importlib.reload(birdclef_audio_processing)
importlib.reload(birdclef_metrics)
importlib.reload(birdclef_datasets)
importlib.reload(birdclef_models)
importlib.reload(birdclef_utils_pkg)

import multiprocessing as mp
mp.set_start_method('spawn', force=True)

In [126]:
# Cell 3a
# Windowed dataset + samplers integration
# Imports for new windowed dataset and samplers
from birdclef_utils.datasets import WindowedAudioDataset, EpochShuffledSampler
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import torch
from torch.nn.utils.rnn import pad_sequence

# Lightweight collate that supports the (spec, target) and (spec, target, rec_idx, start) formats
def collate_windowed(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    if len(batch[0]) == 2:
        specs, targets = zip(*batch)
        rec_idxs, starts = None, None
    else:
        specs, targets, rec_idxs, starts = zip(*batch)
    specs_padded = pad_sequence(specs, batch_first=True, padding_value=0.0)
    targets_stacked = torch.stack(targets)
    return specs_padded, targets_stacked, rec_idxs, starts

# Compute per-window sample weights given per-class weights (class_weights: array of length C)
def compute_window_weights_from_class_weights(dataset, class_weights):
    # dataset.index must be present (windowed modes). dataset.targets indexes by record index.
    if not hasattr(dataset, 'index') or dataset.index is None:
        raise ValueError('Dataset must be windowed (have dataset.index)')
    weights = np.zeros(len(dataset), dtype=float)
    for i, (rec_idx, start) in enumerate(dataset.index):
        t = np.array(dataset.targets[rec_idx], dtype=float)
        weights[i] = float(np.dot(t, class_weights))
    # avoid zeros
    weights = weights + 1e-6
    return weights

# Coverage logger to call at end of epoch: accepts accumulated targets_seen (numpy array length C)
def log_epoch_coverage(targets_seen, labels=None, top_n=10):
    total_seen = targets_seen.sum()
    per_class = targets_seen
    print(f'Per-class seen min/max/total: {per_class.min()}/{per_class.max()}/{total_seen}')
    if labels is not None:
        # list rarest classes
        order = np.argsort(per_class)[:top_n]
        print('Rarest classes:')
        for k in order:
            print(k, labels[k], int(per_class[k]))

# Example helper that builds dataset + loader (does not enable weighted sampler by default)
def build_windowed_loader(file_paths, targets, window_samples, stride, batch_size, num_workers=4, mode='windowed-shuffled', use_weighted_sampler=False, class_weights=None):
    ds = WindowedAudioDataset(file_paths, targets, window_samples=window_samples, stride=stride, mode=mode)
    sampler = None
    if mode.startswith('windowed'):
        sampler = EpochShuffledSampler(ds, seed=42)
    if use_weighted_sampler and class_weights is not None:
        w = compute_window_weights_from_class_weights(ds, class_weights)
        weighted_sampler = WeightedRandomSampler(weights=torch.from_numpy(w).double(), num_samples=len(w), replacement=True)
    else:
        weighted_sampler = None
    loader = DataLoader(ds, batch_size=batch_size, sampler=weighted_sampler if weighted_sampler is not None else sampler, collate_fn=collate_windowed, num_workers=num_workers, pin_memory=True)
    return ds, loader, sampler, weighted_sampler

print('Windowed dataset integration cell loaded (defines WindowedAudioDataset, EpochShuffledSampler, collate_windowed, and helpers).')

Windowed dataset integration cell loaded (defines WindowedAudioDataset, EpochShuffledSampler, collate_windowed, and helpers).


In [127]:
## Cell 3
# load the sample files for sanity check
sample_train_audio, sr = load_audio_ogg(SAMPLE_TRAIN_AUDIO_FILE_PATH, sample_rate=32000)
sample_train_soundscapes_audio, sr = load_audio_ogg(SAMPLE_TRAIN_SOUNDSCAPES_FILE_PATH, sample_rate=32000)
if VERBOSE:
    print(f"Train audio shape: {sample_train_audio.shape}, sample rate: {sr}")
    print(f"Soundscapes audio shape: {sample_train_soundscapes_audio.shape}, sample rate: {sr}")

Train audio shape: (367062,), sample rate: 32000
Soundscapes audio shape: (1920000,), sample rate: 32000


In [128]:
## Cell 4
# First let's load and process the training_audio files
from pathlib import Path

training_files = list(Path(training_file_path).glob('*/*.ogg'))

if VERBOSE:
    for file_path in training_files[:5]:
        print(file_path)

train_description = pd.read_csv(AUDIO_DESCRIPTION_FILE, usecols=['primary_label', 'filename', 'rating', 'secondary_labels'])
train_description['folder'] = training_file_path
train_description['secondary_labels'] = train_description['secondary_labels'].str.replace('[', '', regex=False)
train_description['secondary_labels'] = train_description['secondary_labels'].str.replace(']', '', regex=False)
train_description['secondary_labels'] = train_description['secondary_labels'].str.replace("'", "", regex=False)
train_description['secondary_labels'] = train_description['secondary_labels'].fillna('')

if VERBOSE:
    print("Training Audio Descriptions Head:")
    print(train_description.head())

# all_descriptions = pd.concat([soundscape_description, train_description], ignore_index=True)
# print("All Descriptions:")
# print(all_descriptions.head())


data/train_audio/crebec1/XC119358.ogg
data/train_audio/crebec1/XC602873.ogg
data/train_audio/crebec1/XC729360.ogg
data/train_audio/crebec1/XC844032.ogg
data/train_audio/crebec1/XC611358.ogg
Training Audio Descriptions Head:
  primary_label secondary_labels  rating                 filename  \
0       1161364                      0.0  1161364/iNat1216197.ogg   
1       1161364                      0.0  1161364/iNat1114648.ogg   
2       1161364                      0.0   1161364/iNat810195.ogg   
3       1161364                      0.0   1161364/iNat818781.ogg   
4       1161364                      0.0   1161364/iNat556514.ogg   

              folder  
0  data/train_audio/  
1  data/train_audio/  
2  data/train_audio/  
3  data/train_audio/  
4  data/train_audio/  


In [129]:
## Cell 5
# Some analysis on the training description data
rows_with_secondary = train_description[train_description['secondary_labels'] != '']
if VERBOSE:
    print(f"Rows with secondary label values: {rows_with_secondary['secondary_labels']}")

missing_primary_labels = train_description['primary_label'].isnull().sum() + (train_description['primary_label'] == '').sum()
if VERBOSE:
    print(f"Rows with no value for primary_label: {missing_primary_labels}")

missing_filenames = train_description['filename'].isnull().sum() + (train_description['filename'] == '').sum()
if VERBOSE:
    print(f"Rows with no value for filename: {missing_filenames}")

missing_ratings = train_description['rating'].isnull().sum() + (train_description['rating'] == '').sum()
if VERBOSE:
    print(f"Rows with no value for rating: {missing_ratings}")
    print('Breakdown of rating values:')
    rating_counts = train_description['rating'].value_counts().sort_index()
    rating_counts.index = rating_counts.index.astype(float)
    print(rating_counts)


Rows with secondary label values: 138             compau
169             saffin
239      326272, 67107
240      326272, 67107
241             326272
             ...      
35505          flawar1
35506          grasal3
35508          greant1
35533           grekis
35536          sibtan2
Name: secondary_labels, Length: 4372, dtype: str
Rows with no value for primary_label: 0
Rows with no value for filename: 0
Rows with no value for rating: 0
Breakdown of rating values:
rating
0.0    12849
0.5       22
1.0      147
1.5      120
2.0      598
2.5      518
3.0     2738
3.5     1509
4.0     8018
4.5     2185
5.0     6845
Name: count, dtype: int64


In [130]:
## Cell 6
# Length statistics after rating filter
from birdclef_utils import print_audio_file_stats

# Example usage:
# print_audio_file_stats(filtered_description, load_fn=load_audio_tensor_from_ogg, sample_rate=32000)

# Stats for complete dataset:
# Mean: 1116242 - 34.95 seconds at 32kHz
# Min: 256 - 0.008 seconds at 32kHz
# Max: 220195104 - 6874.84 seconds at 32kHz, or about 1.9 hours
# Under 1 seconds: 370 - these look to be mostly in the '0' rating list, but we should remove these from dataset when we load the files just in case.
# Number over 10 seconds: 14232
# Number over 20 seconds: 10500
# Number over 40 seconds: 5653
# Number over 60 seconds: 3236
# Number over 120 seconds: 938
# Number under 1 second: 1

In [131]:
## Cell 7
# Set up audio processing configuration for PyTorch preprocessing
SAMPLE_RATE = 32000
MAX_AUDIO_SAMPLES = None if MAX_AUDIO_SECONDS is None else int(MAX_AUDIO_SECONDS * SAMPLE_RATE)
RANDOM_CROP_SAMPLES = None if RANDOM_CROP_SECONDS is None else int(RANDOM_CROP_SECONDS * SAMPLE_RATE)
FEATURE_BINS = N_MELS if USE_LOG_MEL else (STFT_FFT_LENGTH // 2 + 1)

if VERBOSE:
    print(f"Audio processing config (Pure PyTorch):")
    print(f"  Sample rate: {SAMPLE_RATE} Hz")
    print(f"  Random crop: {RANDOM_CROP_SECONDS} sec = {RANDOM_CROP_SAMPLES} samples" if RANDOM_CROP_SAMPLES else "  No cropping")
    print(f"  STFT: frame_length={STFT_FRAME_LENGTH}, hop={STFT_FRAME_STEP}")
    print(f"  Mel filterbank: {N_MELS} bins, {MEL_FMIN}-{MEL_FMAX} Hz")
    print(f"  Feature bins: {FEATURE_BINS} ({'log-mel' if USE_LOG_MEL else 'magnitude STFT'})")


Audio processing config (Pure PyTorch):
  Sample rate: 32000 Hz
  Random crop: 10 sec = 320000 samples
  STFT: frame_length=1024, hop=320
  Mel filterbank: 128 bins, 50.0-14000.0 Hz
  Feature bins: 128 (log-mel)


In [132]:
## Cell 8
# Split file_paths and multi-label targets into training and validation sets (80/20 split)
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from birdclef_utils.datasets import AudioSpectrogramDataset, CachedAudioSpectrogramDataset, collate_fn_pad

# Build the dataset pipeline with on-the-fly spectrogram computation
MIN_SAMPLES = 32000  # 1 second at 32kHz
MIN_RATING = 4.0

# Recompute feature bins from current config to avoid stale state when toggling USE_LOG_MEL.
FEATURE_BINS = N_MELS if USE_LOG_MEL else (STFT_FFT_LENGTH // 2 + 1)

# Parse label text into token lists.
# Input examples after cleanup: "a,b,c", "a;b;c", or ""
def parse_label_list(s):
    if pd.isna(s):
        return []
    s = str(s).strip()
    if not s:
        return []
    normalized = s.replace(';', ',')
    return [x.strip() for x in normalized.split(',') if x.strip()]

def split_primary_secondary(tokens):
    # Preserve order while removing duplicates/empties.
    seen = set()
    deduped = []
    for t in tokens:
        t = str(t).strip()
        if not t or t in seen:
            continue
        seen.add(t)
        deduped.append(t)

    if not deduped:
        return None, []

    primary = deduped[0]
    secondary = deduped[1:]
    return primary, secondary

def rows_to_arrays(rows, num_classes, label_to_idx):
    file_paths = np.array([row['file_path'] for row in rows], dtype=object)
    primary = np.array([row['primary_label'] for row in rows], dtype=object)
    targets = np.zeros((len(rows), num_classes), dtype=np.float32)

    for i, row in enumerate(rows):
        for lbl in row['labels']:
            if lbl in label_to_idx:
                targets[i, label_to_idx[lbl]] = 1.0

    return file_paths, targets, primary

def stratified_row_split(rows, test_size, random_state):
    if not rows:
        return [], []

    row_primary = np.array([row['primary_label'] for row in rows], dtype=object)
    primary_counts = pd.Series(row_primary).value_counts()
    rare_primary = primary_counts[primary_counts < 2].index.to_numpy()
    rare_mask = np.isin(row_primary, rare_primary)

    rare_rows = [row for row, is_rare in zip(rows, rare_mask) if is_rare]
    split_rows = [row for row, is_rare in zip(rows, rare_mask) if not is_rare]

    if not split_rows:
        return rare_rows, []

    split_primary = np.array([row['primary_label'] for row in split_rows], dtype=object)

    train_idx, val_idx = train_test_split(
        np.arange(len(split_rows)),
        test_size=test_size,
        random_state=random_state,
        stratify=split_primary,
    )

    train_rows = [split_rows[i] for i in train_idx]
    val_rows = [split_rows[i] for i in val_idx]
    train_rows.extend(rare_rows)
    return train_rows, val_rows

def grouped_soundscape_split(rows, test_size, random_state):
    if not rows:
        return [], []

    grouped = {}
    for row in rows:
        grouped.setdefault(row['group_key'], []).append(row)

    group_items = []
    for group_key, group_rows in grouped.items():
        primary_counts = pd.Series([row['primary_label'] for row in group_rows]).value_counts()
        group_items.append({
            'group_key': group_key,
            'rows': group_rows,
            'stratify_label': primary_counts.index[0],
        })

    if len(group_items) < 2:
        train_rows = [row for item in group_items for row in item['rows']]
        return train_rows, []

    group_labels = np.array([item['stratify_label'] for item in group_items], dtype=object)
    label_counts = pd.Series(group_labels).value_counts()
    rare_labels = label_counts[label_counts < 2].index.to_numpy()
    rare_mask = np.isin(group_labels, rare_labels)

    rare_items = [item for item, is_rare in zip(group_items, rare_mask) if is_rare]
    split_items = [item for item, is_rare in zip(group_items, rare_mask) if not is_rare]

    if not split_items:
        train_rows = [row for item in rare_items for row in item['rows']]
        return train_rows, []

    split_labels = np.array([item['stratify_label'] for item in split_items], dtype=object)

    if len(split_items) < 2:
        train_items = split_items
        val_items = []
    else:
        train_idx, val_idx = train_test_split(
            np.arange(len(split_items)),
            test_size=test_size,
            random_state=random_state,
            stratify=split_labels,
        )
        train_items = [split_items[i] for i in train_idx]
        val_items = [split_items[i] for i in val_idx]

    train_items.extend(rare_items)
    train_rows = [row for item in train_items for row in item['rows']]
    val_rows = [row for item in val_items for row in item['rows']]
    return train_rows, val_rows

# --- Primary train_audio rows (rating-filtered) ---
filtered_description = train_description[train_description['rating'] >= MIN_RATING].copy()
filtered_description['secondary_list'] = filtered_description['secondary_labels'].apply(parse_label_list)

base_rows = []
for _, row in filtered_description.iterrows():
    primary_raw = str(row['primary_label']).strip()
    secondary_raw = row['secondary_list']
    primary, secondary = split_primary_secondary([primary_raw] + secondary_raw)
    if primary is None:
        continue

    labels = [primary] + secondary
    file_path = os.path.join(row['folder'], row['filename'])
    base_rows.append({
        'file_path': file_path,
        'labels': labels,
        'primary_label': primary,
        'source': 'train_audio',
        'group_key': file_path,
    })

# --- Optional train_soundscapes segment rows ---
soundscape_rows = []
if USE_TRAIN_SOUNDSCAPE_LABELS:
    soundscape_df = pd.read_csv(SOUNDSCAPES_DESCRIPTION_FILE)

    # Remove duplicated annotations and invalid rows early.
    soundscape_df = soundscape_df.dropna(subset=['filename', 'start', 'end', 'primary_label']).copy()
    soundscape_df = soundscape_df.drop_duplicates(subset=['filename', 'start', 'end', 'primary_label'])

    for _, row in soundscape_df.iterrows():
        label_tokens = parse_label_list(row['primary_label'])
        primary, secondary = split_primary_secondary(label_tokens)
        if primary is None:
            continue

        start_sec = float(pd.to_timedelta(str(row['start'])).total_seconds())
        end_sec = float(pd.to_timedelta(str(row['end'])).total_seconds())
        if end_sec <= start_sec:
            continue

        base_fp = os.path.join(train_soundscapes_path, str(row['filename']))
        seg_fp = f"{base_fp}__SEG__{start_sec:.3f}__{end_sec:.3f}"
        labels = [primary] + secondary

        soundscape_rows.append({
            'file_path': seg_fp,
            'labels': labels,
            'primary_label': primary,
            'source': 'train_soundscapes',
            'group_key': base_fp,
        })

all_rows = base_rows + soundscape_rows
TRAIN_SOUNDSCAPE_ROWS = len(soundscape_rows)

if len(all_rows) == 0:
    raise ValueError('No training rows available after filtering and optional soundscape merge.')

# Build class vocabulary from all merged labels
all_label_set = set()
for row in all_rows:
    all_label_set.update(row['labels'])

all_labels = sorted(all_label_set)
num_classes = len(all_labels)
label_to_idx = {lbl: idx for idx, lbl in enumerate(all_labels)}
idx_to_label = {idx: lbl for lbl, idx in label_to_idx.items()}

base_train_rows, base_val_rows = stratified_row_split(base_rows, test_size=0.2, random_state=42)
soundscape_train_rows, soundscape_val_rows = grouped_soundscape_split(soundscape_rows, test_size=0.2, random_state=42)

train_rows = base_train_rows + soundscape_train_rows
val_rows = base_val_rows + soundscape_val_rows

train_file_paths, train_targets, train_primary = rows_to_arrays(train_rows, num_classes, label_to_idx)
val_file_paths, val_targets, val_primary = rows_to_arrays(val_rows, num_classes, label_to_idx)

# Shuffle training set after concatenation
perm = np.random.default_rng(42).permutation(len(train_file_paths))
train_file_paths = train_file_paths[perm]
train_targets = train_targets[perm]
train_primary = train_primary[perm]

# Keep merged views available for downstream diagnostics.
file_paths = [row['file_path'] for row in all_rows]
primary_labels = np.array([row['primary_label'] for row in all_rows], dtype=object)
labels_matrix = np.zeros((len(file_paths), num_classes), dtype=np.float32)
for i, row in enumerate(all_rows):
    for lbl in row['labels']:
        if lbl in label_to_idx:
            labels_matrix[i, label_to_idx[lbl]] = 1.0

if VERBOSE:
    print(f"Rows after rating filter (train_audio): {len(base_rows)}")
    print(f"Rows added from train_soundscapes: {len(soundscape_rows)}")
    print(f"Merged rows total: {len(file_paths)}")
    print(f"Number of classes (merged labels): {num_classes}")
    print(f"Feature bins: {FEATURE_BINS} ({'log-mel' if USE_LOG_MEL else 'magnitude STFT'})")

    base_train_count = sum(row['source'] == 'train_audio' for row in train_rows)
    base_val_count = sum(row['source'] == 'train_audio' for row in val_rows)
    soundscape_train_count = sum(row['source'] == 'train_soundscapes' for row in train_rows)
    soundscape_val_count = sum(row['source'] == 'train_soundscapes' for row in val_rows)
    train_soundscape_groups = {row['group_key'] for row in train_rows if row['source'] == 'train_soundscapes'}
    val_soundscape_groups = {row['group_key'] for row in val_rows if row['source'] == 'train_soundscapes'}
    leaked_soundscape_groups = train_soundscape_groups & val_soundscape_groups

    print(f"Train samples: {len(train_file_paths)} | Validation samples: {len(val_file_paths)}")
    print(f"Train composition -> train_audio: {base_train_count}, soundscape segments: {soundscape_train_count}")
    print(f"Val composition -> train_audio: {base_val_count}, soundscape segments: {soundscape_val_count}")
    print(f"Soundscape source files -> train: {len(train_soundscape_groups)}, val: {len(val_soundscape_groups)}, overlap: {len(leaked_soundscape_groups)}")
    if leaked_soundscape_groups:
        raise ValueError(f"Soundscape leakage detected across split: {sorted(leaked_soundscape_groups)[:5]}")

# Per-class positive weighting with square root dampening for severe multi-label imbalance
# Square root dampening: transforms 32→14229 range to approximately 5.7→119
positive_counts = train_targets.sum(axis=0)
negative_counts = len(train_targets) - positive_counts
pos_weights = np.sqrt(negative_counts / np.maximum(positive_counts, 1.0)).astype(np.float32)
pos_weights = np.maximum(pos_weights, 1.0)

if VERBOSE:
    print(f"Positive labels per sample (train mean): {train_targets.sum(axis=1).mean():.3f}")
    print(f"Pos weight range: {pos_weights.min():.2f} to {pos_weights.max():.2f}")


# Create PyTorch datasets
# Use cached version for 3-5x speedup (caches full spectrograms, samples random crops)
DatasetClass = CachedAudioSpectrogramDataset if USE_SPECTROGRAM_CACHE else AudioSpectrogramDataset

dataset_kwargs = {
    'sample_rate': SAMPLE_RATE,
    'n_fft': STFT_FRAME_LENGTH,
    'hop_length': STFT_FRAME_STEP,
    'n_mels': N_MELS,
    'fmin': MEL_FMIN,
    'fmax': MEL_FMAX,
    'crop_samples': RANDOM_CROP_SAMPLES,
    'min_samples': MIN_SAMPLES
}

if USE_SPECTROGRAM_CACHE:
    dataset_kwargs['cache_dir'] = SPECTROGRAM_CACHE_DIR
    if VERBOSE:
        print(f"Using spectrogram cache: {SPECTROGRAM_CACHE_DIR}")

train_dataset = DatasetClass(
    train_file_paths,
    train_targets,
    is_train=True,
    **dataset_kwargs
)

val_dataset = DatasetClass(
    val_file_paths,
    val_targets,
    is_train=False,
    **dataset_kwargs
)

# Create PyTorch dataloaders
BATCH_SIZE = 32  # Reduced to minimize disk I/O bottleneck with NUM_WORKERS=0
NUM_WORKERS = 0  # Single process to eliminate multiprocessing overhead and disk contention

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn_pad,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn_pad,
    pin_memory=True if torch.cuda.is_available() else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

if VERBOSE:
    print(f"Created PyTorch DataLoaders:")
    print(f"  Train: {len(train_dataset)} samples, {len(train_loader)} batches")
    print(f"  Val: {len(val_dataset)} samples, {len(val_loader)} batches")
    print(f"  Batch size: {BATCH_SIZE}")

    sample_spec, sample_target = train_dataset[0]

    print(f"Actual spectrogram shape: {sample_spec.shape}")  # Should be (time, n_mels)

Rows after rating filter (train_audio): 17048
Rows added from train_soundscapes: 739
Merged rows total: 17787
Number of classes (merged labels): 215
Feature bins: 128 (log-mel)
Train samples: 14244 | Validation samples: 3543
Train composition -> train_audio: 13639, soundscape segments: 605
Val composition -> train_audio: 3409, soundscape segments: 134
Soundscape source files -> train: 54, val: 12, overlap: 0
Positive labels per sample (train mean): 1.350
Pos weight range: 5.86 to 119.34
Using spectrogram cache: cache/spectrograms
Created PyTorch DataLoaders:
  Train: 14244 samples, 446 batches
  Val: 3543 samples, 111 batches
  Batch size: 32
Actual spectrogram shape: torch.Size([997, 128])


In [133]:
## Windowed dataset builder: keep disabled for deadline runs
USE_WINDOWED = False
WINDOW_SAMPLES = RANDOM_CROP_SAMPLES if RANDOM_CROP_SAMPLES is not None else int(10 * SAMPLE_RATE)
STRIDE = max(1, WINDOW_SAMPLES // 2)

if USE_WINDOWED:
    ds_train, loader_train, sampler_train, weighted_sampler_train = build_windowed_loader(
        train_file_paths, train_targets,
        window_samples=WINDOW_SAMPLES,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        mode='windowed-shuffled',
        use_weighted_sampler=False,
        class_weights=None,
    )

    ds_val, loader_val, sampler_val, weighted_sampler_val = build_windowed_loader(
        val_file_paths, val_targets,
        window_samples=WINDOW_SAMPLES,
        stride=STRIDE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        mode='windowed-exhaustive',
        use_weighted_sampler=False,
        class_weights=None,
    )

    print(f"Windowed train windows: {len(ds_train)} | Windowed val windows: {len(ds_val)}")

    # Replace DataLoader variables used by training
    train_loader = loader_train
    val_loader = loader_val
    # Expose for debugging
    windowed_dataset_train = ds_train
    windowed_dataset_val = ds_val
else:
    ds_train = None
    ds_val = None
    windowed_dataset_train = None
    windowed_dataset_val = None
    print("Windowed mode disabled; keeping original train_loader/val_loader")

Windowed mode disabled; keeping original train_loader/val_loader


In [134]:
## Cell 8b - Cache diagnostics: verify cache files exist and are being found
import hashlib
from pathlib import Path

if VERBOSE:
    # Test cache lookup for a few samples
    print("=== Cache Diagnostics ===\n")
    print(f"Cache directory: {SPECTROGRAM_CACHE_DIR}")
    print(f"Cache exists: {Path(SPECTROGRAM_CACHE_DIR).exists()}")

    if Path(SPECTROGRAM_CACHE_DIR).exists():
        cache_files = list(Path(SPECTROGRAM_CACHE_DIR).glob("*.npy"))
        print(f"Total cache files on disk: {len(cache_files):,}\n")

    # Check if dataset is using cache
    print(f"Dataset class: {type(train_dataset).__name__}")
    print(f"Using cache: {isinstance(train_dataset, CachedAudioSpectrogramDataset)}\n")

    # Test cache lookup for first 10 samples
    if isinstance(train_dataset, CachedAudioSpectrogramDataset):
        found = 0
        missing = 0
        print("Testing cache lookup for first 10 train samples:")
        for i in range(min(10, len(train_dataset))):
            file_path = train_dataset.file_paths[i]
            cache_path = train_dataset._get_cache_path(file_path)
            exists = cache_path.exists()
            status = "✓ FOUND" if exists else "✗ MISSING"
            print(f"  [{i}] {status}: {Path(file_path).name[:50]}")
            if exists:
                found += 1
            else:
                missing += 1
        
        print(f"\nCache lookup summary: {found} found, {missing} missing")
        
        # Check soundscape segments specifically
        soundscape_count = 0
        soundscape_cached = 0
        print("\nChecking soundscape segments (with __SEG__ tokens):")
        for i in range(len(train_dataset)):
            file_path = str(train_dataset.file_paths[i])
            if '__SEG__' in file_path:
                soundscape_count += 1
                cache_path = train_dataset._get_cache_path(file_path)
                if cache_path.exists():
                    soundscape_cached += 1
                if soundscape_count <= 5:  # Show first 5
                    status = "✓" if cache_path.exists() else "✗"
                    print(f"  {status} {Path(file_path).name[:60]}")
        
        print(f"\nSoundscape segments: {soundscape_cached}/{soundscape_count} cached")
    else:
        print("WARNING: Not using CachedAudioSpectrogramDataset!")

=== Cache Diagnostics ===

Cache directory: cache/spectrograms
Cache exists: True
Total cache files on disk: 19,924

Dataset class: CachedAudioSpectrogramDataset
Using cache: True

Testing cache lookup for first 10 train samples:
  [0] ✓ FOUND: XC63646.ogg
  [1] ✓ FOUND: XC565519.ogg
  [2] ✓ FOUND: XC654059.ogg
  [3] ✓ FOUND: XC492026.ogg
  [4] ✓ FOUND: XC472480.ogg
  [5] ✓ FOUND: XC444243.ogg
  [6] ✓ FOUND: BC2026_Train_0050_S22_20220205_214500.ogg__SEG__40
  [7] ✓ FOUND: XC550048.ogg
  [8] ✓ FOUND: XC1017681.ogg
  [9] ✓ FOUND: XC301357.ogg

Cache lookup summary: 10 found, 0 missing

Checking soundscape segments (with __SEG__ tokens):
  ✓ BC2026_Train_0050_S22_20220205_214500.ogg__SEG__40.000__45.0
  ✓ BC2026_Train_0039_S22_20211231_201500.ogg__SEG__30.000__35.0
  ✓ BC2026_Train_0051_S22_20220208_231500.ogg__SEG__15.000__20.0
  ✓ BC2026_Train_0012_S03_20220211_233000.ogg__SEG__5.000__10.00
  ✓ BC2026_Train_0034_S22_20211222_013000.ogg__SEG__20.000__25.0

Soundscape segments: 605/605 c

In [135]:
## Cell 9
from sklearn.metrics import roc_auc_score
from datetime import datetime, timezone
from pathlib import Path
import time
from birdclef_utils.models import BirdCLEFModel, CryCNNClassifierComplex, BirdClefCNNModel

class WeightedBCELoss(nn.Module):
    def __init__(self, pos_weights):
        super().__init__()
        self.pos_weights = torch.tensor(pos_weights, dtype=torch.float32)
        if torch.cuda.is_available():
            self.pos_weights = self.pos_weights.cuda()
        elif torch.backends.mps.is_available():
            self.pos_weights = self.pos_weights.to('mps')
    
    def forward(self, y_pred, y_true):
        # Clip predictions to avoid log(0)
        y_pred = torch.clamp(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Weighted BCE
        positive_term = -y_true * torch.log(y_pred) * self.pos_weights
        negative_term = -(1.0 - y_true) * torch.log(1.0 - y_pred)
        loss = positive_term + negative_term
        return loss.mean()


class FocalLoss(nn.Module):
    """
    Focal Loss for multi-label classification.
    Focuses on hard examples by down-weighting easy ones.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Class weights (like pos_weights in BCE). Shape: (num_classes,)
        gamma: Focusing parameter. Higher values give more focus to hard examples.
               Typically 2.0. Set to 0 to recover standard weighted BCE.
    """
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.alpha = torch.tensor(alpha, dtype=torch.float32)
        self.gamma = gamma
        
        if torch.cuda.is_available():
            self.alpha = self.alpha.cuda()
        elif torch.backends.mps.is_available():
            self.alpha = self.alpha.to('mps')
    
    def forward(self, y_pred, y_true):
        # Clip predictions to avoid log(0)
        y_pred = torch.clamp(y_pred, 1e-7, 1.0 - 1e-7)
        
        # For positive samples: FL = -alpha * (1 - p)^gamma * log(p)
        # For negative samples: FL = -alpha * p^gamma * log(1 - p)
        
        # Compute p_t (probability of true class)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        
        # Compute focal weight: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        
        # Compute cross-entropy: -log(p_t)
        ce = -torch.log(p_t)
        
        # Apply class weighting (alpha) to positive samples only
        alpha_t = y_true * self.alpha + (1 - y_true) * 1.0
        
        # Final focal loss
        loss = alpha_t * focal_weight * ce
        
        return loss.mean()


def compute_contest_auc(y_true, y_pred):
    # Compute contest-style macro ROC-AUC, skipping classes with no positives or all positives.
    y_true_np = y_true.cpu().numpy() if torch.is_tensor(y_true) else y_true
    y_pred_np = y_pred.cpu().numpy() if torch.is_tensor(y_pred) else y_pred
    
    valid_indices = []
    for class_idx in range(y_true_np.shape[1]):
        y_col = y_true_np[:, class_idx]
        positives = int(np.sum(y_col))
        if positives == 0 or positives == len(y_col):
            continue
        valid_indices.append(class_idx)
    
    if not valid_indices:
        return np.nan, 0, int(y_true_np.shape[1])
    
    score = float(roc_auc_score(
        y_true_np[:, valid_indices],
        y_pred_np[:, valid_indices],
        average='macro',
    ))
    return score, len(valid_indices), int(y_true_np.shape[1] - len(valid_indices))


def train_model_pytorch(
    train_loader,
    val_loader,
    input_size,
    num_classes,
    pos_weights,
    epochs=10,
    lr=0.001,
    checkpoint_path='checkpoints/best_model.pt',
    device='cpu',
    resume_training=False,
    resume_checkpoint_path=None,
    resume_initial_epoch=0,
    verbose=True,
    use_focal_loss=False,
    focal_gamma=2.0,
    contest_eval_enabled=True,
    contest_eval_every_n_epochs=1,
    early_stopping_patience=5,
    early_stopping_min_delta=1e-4,
    model_factory=None,
):
    """Train loop with per-epoch coverage logging.

    This redefinition overrides the previous `train_model_pytorch` when run.
    It preserves the core training/validation flow and adds per-epoch logging of
    how many positive windows were seen per class and optional per-record counts
    if the dataset exposes `file_paths` and batches include record indices.
    """
    import time
    from pathlib import Path
    import numpy as np
    import torch

    # Build model from factory or expect a pre-built model in scope
    if model_factory is None:
        # try to use existing BirdClefCNNModel factory if available
        try:
            model = BirdClefCNNModel(n_mels=input_size, time_steps=997, num_classes=num_classes, dropout=0.2)
        except Exception:
            raise ValueError('No model_factory provided and default model construction failed')
    else:
        model = model_factory(input_size=input_size, num_classes=num_classes)

    model = model.to(device)

    # Loss
    if use_focal_loss:
        criterion = FocalLoss(alpha=pos_weights, gamma=focal_gamma)
    else:
        criterion = WeightedBCELoss(pos_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6)

    best_val_auc = -np.inf
    best_epoch = 0
    patience_counter = 0
    initial_epoch = resume_initial_epoch if resume_training else 0

    history = {
        'epoch_number': [],
        'train_loss': [],
        'val_loss': [],
        'val_auc': [],
        'checkpoint_path': checkpoint_path,
    }

    # Helper: determine if loader yields record indices
    def _unpack_batch(batch):
        # Accept either (specs, targets) or (specs, targets, rec_idxs, starts)
        if batch is None:
            return None, None, None
        if isinstance(batch, (list, tuple)):
            if len(batch) == 2:
                specs, targets = batch
                rec_idxs = None
            elif len(batch) >= 3:
                specs, targets, rec_idxs = batch[0], batch[1], batch[2]
            else:
                raise ValueError('Unexpected batch format')
        else:
            raise ValueError('Unexpected batch container type')
        return specs, targets, rec_idxs

    def _move_optimizer_state_to_device(optim, target_device):
        for state in optim.state.values():
            for key, value in state.items():
                if torch.is_tensor(value):
                    state[key] = value.to(target_device)

    # Optionally precompute record count if available
    dataset = getattr(train_loader, 'dataset', None)
    record_count = None
    if dataset is not None and hasattr(dataset, 'file_paths'):
        try:
            record_count = len(dataset.file_paths)
        except Exception:
            record_count = None

    timestamp_utc = datetime.now(timezone.utc).isoformat()
    Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

    # Resume support: load model + optimizer/scheduler state before first resumed epoch.
    if resume_training:
        if not resume_checkpoint_path:
            raise ValueError('resume_training=True requires resume_checkpoint_path')
        resume_path = Path(resume_checkpoint_path)
        if not resume_path.exists():
            raise FileNotFoundError(f'Resume checkpoint not found: {resume_checkpoint_path}')

        ck = torch.load(resume_path, map_location=device)
        if 'model_state_dict' not in ck:
            raise KeyError(f"Checkpoint missing 'model_state_dict': {resume_checkpoint_path}")

        model.load_state_dict(ck['model_state_dict'])

        if 'optimizer_state_dict' in ck:
            optimizer.load_state_dict(ck['optimizer_state_dict'])
            _move_optimizer_state_to_device(optimizer, device)

        if 'scheduler_state_dict' in ck:
            scheduler.load_state_dict(ck['scheduler_state_dict'])

        best_val_auc = float(ck.get('best_val_auc', best_val_auc))
        best_epoch = int(ck.get('best_epoch', ck.get('epoch', best_epoch)))
        patience_counter = int(ck.get('patience_counter', 0))

        if resume_initial_epoch <= 0:
            initial_epoch = int(ck.get('epoch', 0))

        if verbose:
            print(
                f"[RESUME] Loaded checkpoint: {resume_path} | "
                f"epoch={ck.get('epoch', 'unknown')} | best_val_auc={best_val_auc:.6f}"
            )
            print(f"[RESUME] Continuing at epoch {initial_epoch + 1}")

    for epoch in range(epochs):
        actual_epoch = initial_epoch + epoch + 1
        if verbose:
            print(f"\n=== Epoch {actual_epoch}/{initial_epoch+epochs} ===")

        # Reset epoch counters
        model.train()
        train_loss = 0.0
        train_batches = 0

        seen_per_class = np.zeros(num_classes, dtype=int)
        seen_per_record = np.zeros(record_count, dtype=int) if record_count is not None else None

        report_interval = len(train_loader) // 10 if len(train_loader) >= 10 else len(train_loader) + 1

        # If sampler supports set_epoch, seed it for determinism
        sampler = getattr(train_loader, 'sampler', None)
        if sampler is not None and hasattr(sampler, 'set_epoch'):
            try:
                sampler.set_epoch(epoch)
            except Exception:
                pass

        for batch_idx, batch in enumerate(train_loader):
            specs, targets, rec_idxs = _unpack_batch(batch)
            if specs is None:
                continue

            specs = specs.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(specs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            train_batches += 1

            # Update coverage counters (from targets)
            try:
                t_np = targets.detach().cpu().numpy()
                seen_per_class += t_np.sum(axis=0).astype(int)
            except Exception:
                pass

            # Update per-record counters if indices are available
            if rec_idxs is not None and seen_per_record is not None:
                try:
                    # rec_idxs may be a tuple/list; convert to numpy
                    rr = np.array(rec_idxs, dtype=int)
                    for r in rr:
                        if 0 <= r < len(seen_per_record):
                            seen_per_record[r] += 1
                except Exception:
                    pass

            if verbose and (batch_idx + 1) % report_interval == 0:
                print(f"Epoch {actual_epoch} - Batch {batch_idx+1}/{len(train_loader)} - Loss: {loss.item():.4f}")

        # End of epoch: log coverage
        total_seen = int(seen_per_class.sum())
        print(f"Epoch {actual_epoch} coverage: per-class min/max/total = {seen_per_class.min()}/{seen_per_class.max()}/{total_seen}")
        if seen_per_class.min() == 0:
            rare_idx = np.where(seen_per_class == 0)[0]
            print(f"Classes never seen this epoch: {len(rare_idx)} (showing up to 20): {rare_idx[:20].tolist()}")

        if seen_per_record is not None:
            zero_records = int((seen_per_record == 0).sum())
            print(f"Records with zero windows seen this epoch: {zero_records}/{len(seen_per_record)}")

        avg_train_loss = train_loss / max(1, train_batches)

        # Validation
        model.eval()
        val_loss = 0.0
        val_batches = 0
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for batch in val_loader:
                specs, targets, _ = _unpack_batch(batch) if isinstance(batch, (list, tuple)) else (None, None, None)
                if specs is None:
                    continue
                specs = specs.to(device)
                targets = targets.to(device)
                outputs = model(specs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                val_batches += 1
                all_preds.append(outputs.cpu())
                all_targets.append(targets.cpu())

        avg_val_loss = val_loss / max(1, val_batches)
        val_contest_auc = np.nan
        if len(all_preds) > 0:
            try:
                all_preds_tensor = torch.cat(all_preds, dim=0)
                all_targets_tensor = torch.cat(all_targets, dim=0)
                val_contest_auc, _, _ = compute_contest_auc(all_targets_tensor, all_preds_tensor)
            except Exception:
                val_contest_auc = np.nan

        if verbose:
            print(f"Epoch {actual_epoch} summary: train_loss={avg_train_loss:.4f}, val_loss={avg_val_loss:.4f}, val_auc={val_contest_auc}")

        # Scheduler step
        scheduler.step(avg_val_loss)

        # Checkpoint
        if not np.isnan(val_contest_auc) and val_contest_auc > best_val_auc + early_stopping_min_delta:
            best_val_auc = val_contest_auc
            best_epoch = actual_epoch
            patience_counter = 0
            torch.save({
                'epoch': actual_epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_auc': best_val_auc,
                'best_epoch': best_epoch,
                'patience_counter': patience_counter,
            }, checkpoint_path)
            if verbose:
                print(f"Saved new best checkpoint (val_auc={best_val_auc:.6f})")
        else:
            if not np.isnan(val_contest_auc):
                patience_counter += 1
                if patience_counter >= early_stopping_patience:
                    if verbose:
                        print(f"Early stopping at epoch {actual_epoch} (best epoch {best_epoch}, val_auc={best_val_auc:.6f})")
                    break

        history['epoch_number'].append(actual_epoch)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_contest_auc)

    # Restore best weights if present
    if Path(checkpoint_path).exists():
        ck = torch.load(checkpoint_path, map_location=device)
        try:
            model.load_state_dict(ck['model_state_dict'])
            if verbose:
                print(f"Restored best model from epoch {ck.get('epoch', '?')}")
        except Exception:
            pass

    return model, history

In [136]:
## Cell 10
from datetime import datetime, timezone

torch.manual_seed(42)
np.random.seed(42)

if QUICK_RUN:
    TRAIN_SUBSET = min(QUICK_RUN_SUBSET, len(train_file_paths))
    if 'QUICK_RUN_VAL_SUBSET' in globals():
        VAL_SUBSET = min(QUICK_RUN_VAL_SUBSET, len(val_file_paths))
    else:
        VAL_SUBSET = min(max(200, QUICK_RUN_SUBSET // 5), len(val_file_paths))
    EPOCHS = QUICK_RUN_EPOCHS
    if VERBOSE:
        print(f"[QUICK_RUN] train={TRAIN_SUBSET} rows, val={VAL_SUBSET} rows, epochs={EPOCHS}")
else:
    TRAIN_SUBSET = len(train_file_paths)
    VAL_SUBSET = len(val_file_paths)
    # When resuming, use RESUME_EXTRA_EPOCHS; otherwise use default 5
    if RESUME_TRAINING:
        EPOCHS = RESUME_EXTRA_EPOCHS
    else:
        EPOCHS = globals().get('EPOCHS', 5)
    if VERBOSE:
        print(f"[FULL_RUN] train={TRAIN_SUBSET} rows, val={VAL_SUBSET} rows, epochs={EPOCHS}")

# Challenge-aligned monitor defaults (override in Cell 1 if desired).
CHALLENGE_METRIC_MONITOR = bool(globals().get('CHALLENGE_METRIC_MONITOR', True))
CHALLENGE_EVAL_EVERY_N_EPOCHS = max(1, int(globals().get('CHALLENGE_EVAL_EVERY_N_EPOCHS', 2)))
CHALLENGE_EARLY_STOPPING_PATIENCE_EVALS = int(globals().get('CHALLENGE_EARLY_STOPPING_PATIENCE_EVALS', 2 if QUICK_RUN else 2))
CHALLENGE_EARLY_STOPPING_MIN_DELTA = float(globals().get('CHALLENGE_EARLY_STOPPING_MIN_DELTA', 0.0005))

if VERBOSE:
    print(f"[CHALLENGE_EVAL] enabled={CHALLENGE_METRIC_MONITOR}, every_n_epochs={CHALLENGE_EVAL_EVERY_N_EPOCHS}")
    print(
        f"[EARLY_STOP] patience_evals={CHALLENGE_EARLY_STOPPING_PATIENCE_EVALS} | "
        f"min_delta={CHALLENGE_EARLY_STOPPING_MIN_DELTA}"
    )

# Unique checkpoint path per run (includes seconds to avoid collisions)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
CHECKPOINT_BEST_PATH = f'checkpoints/best_model_{RUN_ID}.pt'

if VERBOSE:
    print(f"[RUN_ID] {RUN_ID}")
    print(f"[CHECKPOINT] {CHECKPOINT_BEST_PATH}")

# Subset DataLoaders for quick runs
if QUICK_RUN:
    # Create subset indices
    train_indices = list(range(min(TRAIN_SUBSET, len(train_dataset))))
    val_indices = list(range(min(VAL_SUBSET, len(val_dataset))))
    
    from torch.utils.data import Subset
    train_subset = Subset(train_dataset, train_indices)
    val_subset = Subset(val_dataset, val_indices)
    
    train_loader_subset = DataLoader(
        train_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_fn_pad,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader_subset = DataLoader(
        val_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn_pad,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    train_loader_to_use = train_loader_subset
    val_loader_to_use = val_loader_subset
    
    if VERBOSE:
        print(f"Using subset loaders: train={len(train_subset)}, val={len(val_subset)}")
else:
    train_loader_to_use = train_loader
    val_loader_to_use = val_loader
    
# Train the model
model, hist = train_model_pytorch(
    train_loader=train_loader_to_use,
    val_loader=val_loader_to_use,
    input_size=FEATURE_BINS,
    num_classes=num_classes,
    pos_weights=pos_weights,
    epochs=EPOCHS,
    lr=0.001,
    checkpoint_path=CHECKPOINT_BEST_PATH,
    device=device,
    verbose=TRAIN_VERBOSE,
    contest_eval_enabled=CHALLENGE_METRIC_MONITOR,
    contest_eval_every_n_epochs=CHALLENGE_EVAL_EVERY_N_EPOCHS,
    early_stopping_patience=CHALLENGE_EARLY_STOPPING_PATIENCE_EVALS,
    early_stopping_min_delta=CHALLENGE_EARLY_STOPPING_MIN_DELTA,
    resume_training=RESUME_TRAINING,
    resume_checkpoint_path=RESUME_CHECKPOINT_PATH if RESUME_TRAINING else None,
    resume_initial_epoch=RESUME_INITIAL_EPOCH if RESUME_TRAINING else 0,
    use_focal_loss=USE_FOCAL_LOSS,
    focal_gamma=FOCAL_GAMMA,
)

if VERBOSE:
    print(f'\nTraining completed!')
    print(f'Best checkpoint path: {CHECKPOINT_BEST_PATH}')
    print(f'Run sizes -> train rows: {TRAIN_SUBSET}, val rows: {VAL_SUBSET}')
    print(f'Epochs run: {len(hist["train_loss"])}')
    
    # Print final metrics
    final_train_loss = hist['train_loss'][-1]
    final_val_loss = hist['val_loss'][-1]
    final_val_auc = hist['val_auc'][-1]
    
    print(f'\nFinal metrics:')
    print(f'  Train loss: {final_train_loss:.4f}')
    print(f'  Val loss: {final_val_loss:.4f}')
    if not np.isnan(final_val_auc):
        print(f'  Val contest AUC: {final_val_auc:.6f}')
    
    # Print best metrics
    best_val_auc_idx = np.nanargmax(hist['val_auc'])
    best_val_auc = hist['val_auc'][best_val_auc_idx]
    best_epoch_num = hist['epoch_number'][best_val_auc_idx]
    if not np.isnan(best_val_auc):
        print(f'\nBest val contest AUC: {best_val_auc:.6f} at epoch {best_epoch_num}')

[FULL_RUN] train=14244 rows, val=3543 rows, epochs=3
[CHALLENGE_EVAL] enabled=True, every_n_epochs=2
[EARLY_STOP] patience_evals=4 | min_delta=0.0005
[RUN_ID] 20260603_234124
[CHECKPOINT] checkpoints/best_model_20260603_234124.pt
[RESUME] Loaded checkpoint: checkpoints/best_model_20260603_231706.pt | epoch=20 | best_val_auc=0.799172
[RESUME] Continuing at epoch 21

=== Epoch 21/23 ===
Epoch 21 - Batch 44/446 - Loss: 0.0425
Epoch 21 - Batch 88/446 - Loss: 0.0900
Epoch 21 - Batch 132/446 - Loss: 0.0794
Epoch 21 - Batch 176/446 - Loss: 0.0451
Epoch 21 - Batch 220/446 - Loss: 0.0438
Epoch 21 - Batch 264/446 - Loss: 0.0868
Epoch 21 - Batch 308/446 - Loss: 0.0461
Epoch 21 - Batch 352/446 - Loss: 0.0498
Epoch 21 - Batch 396/446 - Loss: 0.0478
Epoch 21 - Batch 440/446 - Loss: 0.0428
Epoch 21 coverage: per-class min/max/total = 1/403/19232
Records with zero windows seen this epoch: 14244/14244
Epoch 21 summary: train_loss=0.0676, val_loss=0.0723, val_auc=0.8078196962029975
Saved new best checkp

In [137]:
## Cell 11
# Post-training metric summary using metrics module
from birdclef_utils.metrics import summarize_training_history_pytorch

if 'hist' in globals():
    POST_TRAIN_SUMMARY = summarize_training_history_pytorch(hist, verbose=VERBOSE)

TRAINING SUMMARY
Epochs run: 3
Final train loss: 0.0691
Final val loss: 0.0697
Best val contest AUC: 0.809191 at epoch 23
Final val contest AUC: 0.809191


In [138]:
## Cell 12
# Threshold sweep using metrics module
from birdclef_utils.metrics import run_threshold_sweep_pytorch



# Run threshold sweep if model exists
if 'model' in globals() and 'val_loader_to_use' in globals():
    POST_TRAIN_SWEEP = run_threshold_sweep_pytorch(
        model=model,
        val_loader=val_loader_to_use,
        device=device,
        threshold_start=FAST_SWEEP_THRESHOLD_START if QUICK_RUN else FULL_SWEEP_THRESHOLD_START,
        threshold_stop=FAST_SWEEP_THRESHOLD_STOP if QUICK_RUN else FULL_SWEEP_THRESHOLD_STOP,
        threshold_step=FAST_SWEEP_THRESHOLD_STEP if QUICK_RUN else FULL_SWEEP_THRESHOLD_STEP,
        max_samples=FAST_SWEEP_MAX_ROWS if QUICK_RUN else FULL_SWEEP_MAX_ROWS,
        verbose=VERBOSE,
    )
    
    # Backward-compatible globals
    results_arr = POST_TRAIN_SWEEP['results_arr']
    results = [tuple(row) for row in results_arr]
    best_t = POST_TRAIN_SWEEP['best_t']
    best_p = POST_TRAIN_SWEEP['best_p']
    best_r = POST_TRAIN_SWEEP['best_r']
    best_f1 = POST_TRAIN_SWEEP['best_f1']

  Processed 320 samples...
  Processed 640 samples...
  Processed 960 samples...
Running threshold sweep on 1024 samples...

Threshold sweep results:
  Best F1: 0.0703 at threshold 0.35
  Precision: 0.0472, Recall: 0.1373


In [ ]:
## Cell 13 - Visualize training curves
from birdclef_utils.metrics import plot_training_curves_and_confusion

# Plot training curves if history exists
if 'hist' in globals() and 'model' in globals():
    # Note: The confusion matrix portion of this function is designed for 
    # single-label classification and may not work correctly for multi-label.
    # The training curves (loss, AUC) will display correctly.
    
    # Prepare training_curves dict in expected format
    training_curves = {
        'train_loss': hist['train_loss'],
        'val_loss': hist['val_loss'],
        'val_auc': hist['val_auc'],
    }
    
    if VERBOSE:
        print("Plotting training curves...")
        print(f"Available metrics: {list(training_curves.keys())}")
    
    # Call plotting function
    # Note: confusion matrix is for single-label, will skip it for multi-label task
    try:
        fig = plot_training_curves_and_confusion(
            training_curves=training_curves,
            model=model,
            dataloader=val_loader_to_use if 'val_loader_to_use' in globals() else val_loader,
            label_to_idx=label_to_idx,
            idx_to_label=idx_to_label,
            device=device,
            phases=['train', 'val'],
            metrics=['loss', 'auc']
        )
    except Exception as e:
        if VERBOSE:
            print(f"Warning: Plotting encountered an error (expected for multi-label tasks): {e}")
            print("Plotting training curves only (without confusion matrix)...")
        
        # Fallback: plot just the curves without confusion matrix
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        epochs = list(range(len(training_curves['train_loss'])))
        
        # Loss plot
        axes[0].plot(epochs, training_curves['train_loss'], label='train', marker='o')
        axes[0].plot(epochs, training_curves['val_loss'], label='val', marker='s')
        axes[0].set_title('Training Curves - Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # AUC plot
        axes[1].plot(epochs, training_curves['val_auc'], label='val', marker='s', color='green')
        axes[1].set_title('Training Curves - AUC')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Contest AUC')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("No training history available. Train a model first (Cell 10).")

In [140]:
## Cell 14
# Append run metrics to CSV using metrics module
from birdclef_utils.metrics import append_run_metrics_csv_pytorch

# Append metrics if history exists
if 'hist' in globals():
    # Build run context
    # Get checkpoint path from history (not stale globals)
    hist_checkpoint = hist.get('checkpoint_path', CHECKPOINT_BEST_PATH if 'CHECKPOINT_BEST_PATH' in globals() else None)
    hist_run_id = Path(hist_checkpoint).stem.replace('best_model_', '') if hist_checkpoint else (RUN_ID if 'RUN_ID' in globals() else 'unknown')
    
    run_context = {
        'run_id': hist_run_id,
        'checkpoint_best_path': hist_checkpoint,
        'model_type': model.model_name if 'model' in globals() else hist.get('model_name', 'unknown'),
        'quick_run': QUICK_RUN,
        'train_subset_rows': TRAIN_SUBSET if 'TRAIN_SUBSET' in globals() else len(train_file_paths),
        'val_subset_rows': VAL_SUBSET if 'VAL_SUBSET' in globals() else len(val_file_paths),
        'train_rows_full_split': len(train_file_paths),
        'val_rows_full_split': len(val_file_paths),
        'num_classes': num_classes,
        'batch_size': BATCH_SIZE,
        'num_workers': NUM_WORKERS,
        'use_spectrogram_cache': USE_SPECTROGRAM_CACHE,
        'feature_bins': FEATURE_BINS,
        'use_log_mel': USE_LOG_MEL,
        'use_focal_loss': USE_FOCAL_LOSS,
        'focal_gamma': FOCAL_GAMMA if USE_FOCAL_LOSS else None,
        'run_tag': globals().get('RUN_TAG', ''),
    }
    
    POST_TRAIN_RUN_LOG_DF = append_run_metrics_csv_pytorch(
        hist,
        run_context=run_context,
        threshold_summary=POST_TRAIN_SWEEP if 'POST_TRAIN_SWEEP' in globals() else None,
        run_log_path='results/run_comparison.csv',
        verbose=TRAIN_VERBOSE,
    )

Saved run metrics to: results/run_comparison.csv
             run_id        model_name  train_subset_rows  val_subset_rows  \
32  20260603_225808  BirdClefCNNModel              14244             3543   
33  20260603_231706  BirdClefCNNModel              14244             3543   
34  20260603_234124  BirdClefCNNModel              14244             3543   

    batch_size  epochs_ran  best_epoch  best_val_contest_auc  \
32          32           3          17              0.788101   
33          32           3          20              0.799172   
34          32           3          23              0.809191   

    last_val_contest_auc  best_val_loss  ...  num_classes  feature_bins  \
32              0.787256       0.073531  ...          215           128   
33              0.773536       0.072483  ...          215           128   
34              0.809191       0.069706  ...          215           128   

    use_log_mel  last_loss  threshold_best threshold_f1  \
32         True   0.07043

In [141]:
## Cell 15
# Contest-metric verification using metrics module
from pathlib import Path
from birdclef_utils.metrics import collect_predictions_pytorch, contest_macro_roc_auc_pytorch
from birdclef_utils.models import BirdCLEFModel

# Fast defaults for routine checks
VERIFY_EVAL_RUN_FULL = bool(globals().get('VERIFY_EVAL_RUN_FULL', False))
VERIFY_EVAL_ROWS = int(globals().get('VERIFY_EVAL_ROWS', min(VALIDATION_EVAL_ROWS, 300)))
VERIFY_EVAL_CROPS = int(globals().get('VERIFY_EVAL_CROPS', 1))
VERIFY_EVAL_BATCH_SIZE = int(globals().get('VERIFY_EVAL_BATCH_SIZE', max(BATCH_SIZE, 32)))

# Run verification if we have a trained model
if 'model' in globals() and 'val_loader' in globals():
    if VERIFY_EVAL_RUN_FULL:
        eval_samples = None  # Use all
        verify_loader = val_loader
    else:
        eval_samples = VERIFY_EVAL_ROWS
        verify_loader = val_loader_to_use if 'val_loader_to_use' in globals() else val_loader
    
    if VERBOSE:
        print('Contest-metric verification for current model')
        print(f"Verify config -> max_samples={eval_samples}, full={VERIFY_EVAL_RUN_FULL}")
    
    y_true_verify, y_pred_verify = collect_predictions_pytorch(
        model,
        verify_loader,
        device=device,
        max_samples=eval_samples,
        verbose=VERBOSE,
    )
    
    CONTEST_EVAL_VERIFY = contest_macro_roc_auc_pytorch(y_true_verify, y_pred_verify)
    
    print('\nContest-style macro ROC-AUC verification')
    print(
        f"verify_auc={CONTEST_EVAL_VERIFY['macro_roc_auc']:.6f} | "
        f"samples={len(y_true_verify)} | "
        f"classes_scored={CONTEST_EVAL_VERIFY['classes_scored']} | "
        f"skipped_no_positive={CONTEST_EVAL_VERIFY['classes_skipped_no_positive']} | "
        f"skipped_all_positive={CONTEST_EVAL_VERIFY['classes_skipped_all_positive']}"
    )
    
    # Compare with training history if available
    if 'hist' in globals() and 'val_auc' in hist:
        val_aucs = [x for x in hist['val_auc'] if not np.isnan(x)]
        if val_aucs:
            best_train_auc = max(val_aucs)
            print(f"best_train_val_contest_auc={best_train_auc:.6f} | verify_minus_train={CONTEST_EVAL_VERIFY['macro_roc_auc'] - best_train_auc:+.6f}")
else:
    print("No model available for verification. Train a model first (Cell 10).")

Contest-metric verification for current model
Verify config -> max_samples=300, full=False
  Processed 320 samples...

Contest-style macro ROC-AUC verification
verify_auc=0.799594 | samples=320 | classes_scored=129 | skipped_no_positive=86 | skipped_all_positive=0
best_train_val_contest_auc=0.809191 | verify_minus_train=-0.009597


In [142]:
# # Debug DataLoader worker exits: recreate windowed loader with num_workers=0 to get full traceback
# import traceback

# # Rebuild datasets/loaders in main process to capture exceptions from dataset/collate
# ds_train, loader_train, sampler_train, weighted_sampler_train = birdclef_datasets.build_windowed_loader(
#     train_file_paths, train_targets,
#     window_samples=WINDOW_SAMPLES,
#     stride=STRIDE,
#     batch_size=BATCH_SIZE,
#     num_workers=2,  # run in-main-process to get full traceback
#     mode='windowed-shuffled',
#     use_weighted_sampler=False,
#     class_weights=None,
#     cache_dir=SPECTROGRAM_CACHE_DIR,
#  )
# print('Dataset length:', len(ds_train))

# # Try one __getitem__ call in main process to catch exceptions from dataset logic
# try:
#     item = ds_train[0]
#     print('ds_train[0] ->', type(item), ('len=' + str(len(item)) if hasattr(item, '__len__') else ''))
# except Exception:
#     print('Exception during ds_train[0]:')
#     traceback.print_exc()

# # Try fetching one batch from the loader (this will exercise collate as well)
# try:
#     it = iter(loader_train)
#     batch = next(it)
#     print('Fetched batch OK; batch types/shapes:')
#     if batch is None:
#         print('batch is None')
#     else:
#         for i, elem in enumerate(batch):
#             if isinstance(elem, (list, tuple)):
#                 try:
#                     l = len(elem)
#                 except Exception:
#                     l = 'unknown'
#                 print(f'  elem[{i}] type=list/tuple len={l}')
#             elif hasattr(elem, 'shape'):
#                 print(f'  elem[{i}] tensor shape={getattr(elem, "shape", None)}')
#             else:
#                 print(f'  elem[{i}] type={type(elem)}')
# except Exception:
#     print('Exception fetching batch from loader:')
#     traceback.print_exc()

# # If those succeed, scan first N dataset indices to find any index that raises
# print('\nScanning first 1000 dataset indices for exception...')
# bad_found = False
# for i in range(min(1000, len(ds_train))):
#     try:
#         _ = ds_train[i]
#     except Exception:
#         print(f'Exception at dataset index {i}:')
#         traceback.print_exc()
#         bad_found = True
#         break
# if not bad_found:
#     print('No per-index exceptions in first 200 items; increase scan range if needed')

# # Hint: if issue persists with num_workers>0, try setting start method to spawn BEFORE creating loaders:
# print("\nIf problem persists when using num_workers>0, run this once at kernel start before creating DataLoaders:")
# print("  import multiprocessing as mp; mp.set_start_method('spawn', force=True)")


In [144]:
# # Quick diagnostics (run in notebook)
# import os
# from birdclef_utils import datasets as birdclef_datasets

# print('file_paths len:', len(file_paths))
# print('first 5 paths:', file_paths[:5])
# print('targets len:', len(targets))

# # Do files actually exist?
# for p in file_paths[:5]:
#     print('exists:', os.path.exists(p), p)

# # Build dataset with num_workers=0 (no workers) to inspect internals
# ds = birdclef_datasets.WindowedAudioDataset(file_paths, targets,
#     window_samples=32000, stride=32000, precompute_lengths=True, cache_dir='cache/spectrograms')
# print('window_samples,stride,crop_frames:', ds.window_samples, ds.stride, ds.crop_frames)
# print('lengths sample:', ds.lengths[:20])
# print('index length:', 0 if ds.index is None else len(ds.index))

# # Force length computation by disabling precompute_lengths (will attempt to open files)
# ds2 = birdclef_datasets.WindowedAudioDataset(file_paths, targets,
#     window_samples=32000, stride=32000, precompute_lengths=False, cache_dir='cache/spectrograms')
# print('index length (forced):', 0 if ds2.index is None else len(ds2.index))